In [13]:
# Word analysis is based on the observation that in English text, certain words appear with much higher frequency than others. 
# For example: "THE", "AND", "OF", "TO", "IN", etc.
import requests
import re
import math
import random
from collections import Counter
character_list = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", 
                  "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z", " "]

def build_word_frequency_dict(text, min_word_length=2):
    words = text.split()
    word_freq = {}
    for word in words:
        if len(word) >= min_word_length: # Only consider words length greater than 2
            word_freq[word] = word_freq.get(word, 0) + 1
    total_words = sum(word_freq.values())
    
    # Display statistics
    print(f"Built dictionary with {len(word_freq)} unique words")
    print("Top 10 most frequent words:")
    sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]
    for word, freq in sorted_words:
        probability = freq / total_words
        print(f"  '{word}': {freq} occurrences ({probability:.4f})")
    
    return word_freq, total_words


In [14]:
# Calculate text score based on word frequencies
def word_based_score(text, word_freq_dict, total_words, penalty_weight=2.0):
    score = 0
    words = text.split()
    
    if len(words) == 0:
        return float('-inf')  # Empty text gets worst score
    
    known_words = 0
    unknown_words = 0
    
    for word in words:
        if word in word_freq_dict:
            # Known word: positive contribution based on frequency
            probability = word_freq_dict[word] / total_words
            score += math.log(probability + 1e-10)  # Smoothing to avoid log(0)
            known_words += 1
        else:
            # Unknown word: apply penalty
            score += math.log(1e-10) * penalty_weight
            unknown_words += 1
    
    # Additional reward for high percentage of known words
    if len(words) > 0:
        known_ratio = known_words / len(words)
        score += math.log(known_ratio + 1e-10) * 3
    
    return score

In [15]:
# Combined scoring using both bigram and word frequencies
def combined_score(text, bigram_freq_dicts, word_freq_dict, total_words, 
                  alpha=0.7, verbose=False):
    # Calculate individual scores
    bigram_score = score(text, bigram_freq_dicts)
    word_score = word_based_score(text, word_freq_dict, total_words)
    
    # Normalize by text length
    text_length = max(len(text), 1)  # Avoid division by zero
    normalized_bigram = bigram_score / text_length
    normalized_word = word_score / text_length
    
    # Combine scores
    combined = alpha * normalized_bigram + (1 - alpha) * normalized_word
    
    if verbose and len(text.split()) > 5:
        known_words = sum(1 for w in text.split() if w in word_freq_dict)
        total_words_in_text = len(text.split())
        print(f"Bigram: {bigram_score:7.2f} | Word: {word_score:7.2f} | "
              f"Known: {known_words}/{total_words_in_text} | Combined: {combined:.4f}")
    
    return combined

In [21]:
# Test
message_code = "WDYRDYLDQCSLR KTYDPYZ LXSKTYWDYQ U KTYZOITYGIDYCDYJSILPTYLDUDKOTYCIQYKQTP QPYODYPDRZTYCDQPDQCLDYODTYJL GIDRDQPTYSLH QKGIDTYCDTYMSKTDLKDTYCSIULKLYODTYADINYZSILYXKNDLYODYB ODKCSTJSZDYCDYOSMTJILKPDYCDYHSYPDLYHLYJDYIQDYOIDILYRSRDQP QYDYCDYJSQTJKDQJDYODYTSRRDKOYSIYDP KDQPYZOSQHYTYODTYRDIMODTYO YJF RMLDYODYPSIPYCSQPYWDYQDP KTYGIIQDYZDPKPDYZ LPKDYDPY YOKQTDQTKMKOKPDYCIGIDOYWDYLDPSILQ KTYUKPDYRIQKL"
word_freq_dict, total_words = build_word_frequency_dict(message_code)

def demonstrate_word_analysis():
    
    for description, test_text in test_cases:
        print(f"\n{description}:")
        print(f"Text: '{test_text}'")
        
        bigram_sc = score(test_text, bigram_stats)
        word_sc = word_based_score(test_text, word_freq_dict, total_words)
        combined_sc = combined_score(test_text, bigram_stats, word_freq_dict, total_words)
        
        words = test_text.split()
        known_count = sum(1 for w in words if w in word_freq_dict)
        
        print(f"  Bigram score: {bigram_sc:8.2f}")
        print(f"  Word score:   {word_sc:8.2f}")
        print(f"  Combined:     {combined_sc:8.4f}")
        print(f"  Known words:  {known_count}/{len(words)}")

Built dictionary with 16 unique words
Top 10 most frequent words:
  'WDYRDYLDQCSLR': 1 occurrences (0.0625)
  'KTYDPYZ': 1 occurrences (0.0625)
  'LXSKTYWDYQ': 1 occurrences (0.0625)
  'KTYZOITYGIDYCDYJSILPTYLDUDKOTYCIQYKQTP': 1 occurrences (0.0625)
  'QPYODYPDRZTYCDQPDQCLDYODTYJL': 1 occurrences (0.0625)
  'GIDRDQPTYSLH': 1 occurrences (0.0625)
  'QKGIDTYCDTYMSKTDLKDTYCSIULKLYODTYADINYZSILYXKNDLYODYB': 1 occurrences (0.0625)
  'ODKCSTJSZDYCDYOSMTJILKPDYCDYHSYPDLYHLYJDYIQDYOIDILYRSRDQP': 1 occurrences (0.0625)
  'QYDYCDYJSQTJKDQJDYODYTSRRDKOYSIYDP': 1 occurrences (0.0625)
  'KDQPYZOSQHYTYODTYRDIMODTYO': 1 occurrences (0.0625)
